# 05 · 张量自动求导（一）：Tensor 与广播反向

> **本节属于 Part 3 · 张量自动求导。这是 minitorch 真正的"心脏"。**

Part 2 的 `Value` 把每个**标量**当作图节点——正确，但太慢。本节我们把 autograd 升级到**张量**：一个 `Tensor` 节点包住一整个 NumPy 数组，用向量化一次算完一层。

本节先打地基：实现 `Tensor` 的骨架、逐元素运算，并攻克最容易出错的一关——**广播 (broadcasting) 的反向传播**。

## 学习目标

- 实现 `Tensor` 骨架：包住 `np.ndarray`，带 `grad / _backward / _prev`
- 实现逐元素 `+`、`*` 与归约 `sum`，并支持 `backward()`
- **彻底理解广播的反向**：核心工具函数 `_unbroadcast`
- 用这个最小引擎**向量化地**重做线性回归（对比 nb01 的手推梯度）

## 直觉与数学原理：广播的反向

NumPy 的**广播**让形状不同的数组也能相加/相乘，例如 `(2,3) + (3,)`：那个 `(3,)` 会被"复制"到每一行。

前向时广播很方便；但**反向时要小心**：如果一个张量在前向被复制了 $k$ 份参与计算，那么它收到的梯度也来自这 $k$ 条路径，必须**求和**回它原来的形状。

所以我们需要一个 `_unbroadcast(grad, shape)`：把广播放大后的梯度，沿"被广播出来的维度"求和，还原成原始 `shape`。这是张量 autograd 最关键、也最容易写错的一处。

In [ ]:
import numpy as np

def _unbroadcast(grad, shape):
    """把 grad 沿"被广播出来的维度"求和，还原成 shape。"""
    # 1) 去掉前面多出来的维度（例如 (2,3) 对 (3,)，多了第 0 维）
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    # 2) 对原本大小为 1、被广播放大的维度求和
    for i, dim in enumerate(shape):
        if dim == 1 and grad.shape[i] != 1:
            grad = grad.sum(axis=i, keepdims=True)
    return grad

# 小例子：原形状 (3,) 被广播成 (2,3)，反向时梯度要按行求和回 (3,)
g = np.ones((2, 3))
print("还原到 (3,):", _unbroadcast(g, (3,)))      # [2, 2, 2]
print("还原到 (1,3):", _unbroadcast(g, (1, 3)))

## 从零手写实现：最小 Tensor

下面这个 `Tensor` 只有 `+`、`*`、`sum` 三个运算，但已经包含了张量 autograd 的全部精髓——注意每个运算的 `_backward` 都用上了 `_unbroadcast`。

In [ ]:
class Tensor:
    def __init__(self, data, _children=()):
        self.data = np.array(data, dtype=np.float64)
        self.grad = np.zeros_like(self.data)
        self._backward = lambda: None
        self._prev = set(_children)

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, (self, other))
        def _backward():
            self.grad  += _unbroadcast(out.grad, self.data.shape)
            other.grad += _unbroadcast(out.grad, other.data.shape)
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, (self, other))
        def _backward():
            self.grad  += _unbroadcast(out.grad * other.data, self.data.shape)
            other.grad += _unbroadcast(out.grad * self.data, other.data.shape)
        out._backward = _backward
        return out

    def sum(self, axis=None, keepdims=False):
        out = Tensor(self.data.sum(axis=axis, keepdims=keepdims), (self,))
        def _backward():
            grad = out.grad
            if axis is not None and not keepdims:
                grad = np.expand_dims(grad, axis)
            self.grad += np.broadcast_to(grad, self.data.shape)
        out._backward = _backward
        return out

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for c in v._prev:
                    build(c)
                topo.append(v)
        build(self)
        self.grad = np.ones_like(self.data)
        for v in reversed(topo):
            v._backward()

    def __neg__(self):   return self * -1
    def __sub__(self, o):return self + (-(o if isinstance(o, Tensor) else Tensor(o)))
    def __repr__(self):  return f"Tensor(shape={self.data.shape})"

## 验证：广播反向对不对？

构造一个带广播的表达式，用**数值梯度检查**和 **PyTorch** 双重验证。

In [ ]:
from minitorch import numerical_gradient, rel_error

A = np.random.randn(2, 3)
b = np.random.randn(3)            # 将被广播

ta, tb = Tensor(A), Tensor(b)
((ta + tb) * ta).sum().backward()  # 一个混合了广播与逐元素乘的表达式

# 数值梯度
ga = numerical_gradient(lambda x: ((x + b) * x).sum(), A.copy())
gb = numerical_gradient(lambda x: ((A + x) * A).sum(), b.copy())
print("A.grad 相对误差:", rel_error(ta.grad, ga))
print("b.grad 相对误差:", rel_error(tb.grad, gb))
print("b.grad 形状还原正确:", tb.grad.shape == b.shape)

In [ ]:
import torch
At = torch.tensor(A, requires_grad=True)
bt = torch.tensor(b, requires_grad=True)
((At + bt) * At).sum().backward()
print("vs PyTorch A:", rel_error(ta.grad, At.grad.numpy()))
print("vs PyTorch b:", rel_error(tb.grad, bt.grad.numpy()))

## 立竿见影：向量化的线性回归

还记得 nb01 里我们**手推**了线性回归的梯度吗？现在有了 `Tensor`，我们只写**前向**，梯度全自动——而且是**向量化**的，一次处理所有样本。

In [ ]:
from minitorch import set_seed
import matplotlib.pyplot as plt

set_seed(42)
x = np.random.uniform(-3, 3, size=100)
y = 2.0 * x - 1.0 + np.random.randn(100) * 0.5
N = len(x)

w, b = Tensor(0.0), Tensor(0.0)
history = []
for epoch in range(300):
    w.grad = np.zeros_like(w.data)        # 梯度清零（w, b 被反复使用）
    b.grad = np.zeros_like(b.data)
    yhat = Tensor(x) * w + b              # 广播：标量 w,b 作用到整个向量
    diff = yhat - Tensor(y)
    loss = (diff * diff).sum() * (1.0 / N)   # MSE
    loss.backward()                        # 自动求梯度！
    w.data -= 0.05 * w.grad
    b.data -= 0.05 * b.grad
    history.append(float(loss.data))

print(f"学到: w={float(w.data):.3f}, b={float(b.data):.3f}  (真值 2, -1)")
plt.figure(figsize=(5,3)); plt.plot(history); plt.yscale("log")
plt.title("Loss (vectorized Tensor autograd)"); plt.xlabel("epoch"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 📦 沉淀进 minitorch

这个 `Tensor` 还很"迷你"。接下来两节我们会补齐 **matmul、各种激活、归约、形状变换**，以及完善反向引擎（`no_grad`、`detach`、梯度累加）。等 **nb07** 全部就绪后，完整版会沉淀进 **`minitorch/tensor.py`**——后续所有 notebook 直接 `from minitorch import Tensor`。

（剧透：包里的完整 `Tensor` 已经写好并通过了全部梯度检查测试，本节只是带你亲手搭出它的地基。）

## 小练习

1. **给 Tensor 加 `mean`**：仿照 `sum` 实现 `mean(axis=None)`（提示：前向用 `data.mean`，反向在 `sum` 的基础上再除以元素个数），并用 `gradcheck` 验证。
2. **更刁钻的广播**：验证 `(4,1) + (1,5) -> (4,5)` 的反向，两个输入的 `grad` 形状应分别还原回 `(4,1)` 和 `(1,5)`。
3. **思考**：为什么 `_backward` 里对 `self.grad` 用 `+=` 而不是 `=`？（提示：一个张量可能被用在多个地方。）

## 小结 & 下一站

✅ 我们搭出了张量 autograd 的地基：`Tensor` 骨架 + 逐元素运算 + **广播反向 `_unbroadcast`**，并用它向量化地训练了线性回归。

✅ 关键洞见：**广播在前向是"复制"，在反向就是"求和还原"**。

**下一站 → `06_matmul_and_more_ops`**：补齐神经网络必需的 `matmul`（含它漂亮的反向公式 $dA=dC\,B^\top,\ dB=A^\top dC$）、各种激活函数、归约与形状变换，并逐一用 `gradcheck` 验收。